In [6]:
# 导入必要的包和模块
from deabook import MB  # 导入MB分析模块
from deabook.constant import RTS_CRS, RTS_VRS1  # 导入规模报酬假设常量
import pandas as pd  # 导入数据处理库

# 设置分类变量的类型和顺序
from pandas.api.types import CategoricalDtype
# 定义效率分解指标的有序分类类型
cat_size_order = CategoricalDtype(
    ['OE', 'TE', 'PIAE', 'NPIAE', 'NPOAE'],  # 效率指标：
    # OE: 整体效率
    # TE: 技术效率  
    # PIAE: 投入配置效率
    # NPIAE: 非投入配置效率
    # NPOAE: 非产出配置效率
    ordered=True
)

# 读取电厂数据并预处理
data = pd.read_excel(r"../../data\power plant data.xlsx").query('year>2014').reset_index(drop=True)  # 读取电厂数据
      # 筛选2014年以后的数据
      # 重置索引

# 重命名数据列（中英文对照）
data.columns = ['province', 'K', 'L', 'F', 'E', 'CO2', 'year', '省份']
# K: 资本投入
# L: 劳动投入  
# F: 燃料投入
# E: 能源投入
# CO2: 二氧化碳排放

dmuname = '省份'  # 设置决策单元名称列为"省份"

# 提取关键变量
yearid = data[['year', dmuname, 'CO2']]  # 包含年份、省份和CO2排放的数据
yearid[dmuname].unique()  # 查看所有唯一的省份名称

# 生成描述性统计表并导出
data[['K','L','F','E','CO2']] .describe().T.to_markdown("table9.1.md")  # 计算描述性统计并转置（行变列）
      # 保存为Markdown格式表格

In [7]:
data

,province,K,L,F,E,CO2,year,省份
0,Beijing,965,5.76,908.31,412.53,1592.34,2015,北京
1,Tianjin,1283,3.27,1920.51,635.88,5115.45,2015,天津
2,Hebei,4350,14.45,7461.84,2288.70,24012.84,2015,河北
3,Shanxi,5940,8.41,7960.77,2318.60,22033.72,2015,山西
4,Inner Mongolia,7263,11.92,14092.70,3421.93,39424.28,2015,内蒙古
...,...,...,...,...,...,...,...,...
85,Shaanxi,3144,9.78,4601.91,1586.05,12534.67,2017,陕西
86,Gansu,2059,11.09,2323.90,713.91,6443.05,2017,甘肃
87,Qinghai,399,2.12,538.12,161.19,1428.51,2017,青海
88,Ningxia,2583,2.91,4163.78,1167.76,11625.40,2017,宁夏


In [8]:
data[['K','L','F','E','CO2']].describe().T

,count,mean,std,min,25%,50%,75%,max
K,90.0,3522.466667,2511.403020,318.00,1778.5000,2579.500,5018.5000,10335.00
L,90.0,10.016667,5.470879,1.41,6.8325,9.620,12.5025,24.08
F,90.0,4860.465222,4010.582299,436.98,2022.8750,3123.490,7396.2275,15874.43
E,90.0,1489.553333,1231.745647,122.00,613.9900,1006.485,2310.3025,5142.88
CO2,90.0,13526.000000,11263.275788,1156.22,5519.7750,8715.520,20544.7675,44495.77


## MB

In [9]:
sx = data['CO2'].sum()/data['F'].sum()  # 总CO2排放/总燃料消耗
sy = 0  # 产出方向调整系数设为0
# 定义并求解DEA模型
def MB_VRSL1():

    # model = DEA.DEA(data,sent = "K L E=Y:CO2",  orient="L", rts=RTS_VRS, baseindex=None,refindex=None)
    model =MB.MB(data,sent = "K L+F=E:CO2", sx=[[0,0,sx]], sy=[[sy]], \
                    rts=RTS_VRS1,baseindex=None,refindex=None,level=1)
    res, info = model.optimize("mosek", "1")  # 使用Mosek求解器
    return res
MB_VRSL1 = MB_VRSL1()  # 执行模型
MB_VRSL1

,optimization_status,obj,best of Undesirable
0,1.0,1592.340000,1592.340000
1,1.0,4794.003224,4794.003224
2,1.0,21825.209573,21825.209573
3,1.0,17951.812421,17951.812421
4,1.0,27910.765567,27910.765567
...,...,...,...
85,1.0,12202.430819,12202.430819
86,1.0,4923.135529,4923.135529
87,1.0,1336.275945,1336.275945
88,1.0,11625.400000,11625.400000


In [10]:
sx = data['CO2'].sum()/data['F'].sum()  # 总CO2排放/总燃料消耗
sy = 0  # 产出方向调整系数设为0
# 定义并求解DEA模型
def MB_CRSL2():

    # model = DEA.DEA(data,sent = "K L E=Y:CO2",  orient="L", rts=RTS_VRS, baseindex=None,refindex=None)
    model =MB.MB(data,sent = "K L+F=E:CO2", sx=[[0,0,sx]], sy=[[sy]], \
                    rts=RTS_CRS,baseindex=None,refindex=None,level=2)
    res, info = model.optimize("mosek", "1")  # 使用Mosek求解器
    return res
MB_CRSL2 = MB_CRSL2()
MB_CRSL2

,optimization_status,obj,best of Undesirable
0,1.0,1592.340000,1592.340000
1,1.0,4752.262658,4752.262658
2,1.0,21822.691755,21822.691755
3,1.0,17904.474658,17904.474658
4,1.0,27513.058805,27513.058805
...,...,...,...
85,1.0,12046.319224,12046.319224
86,1.0,4295.377920,4295.377920
87,1.0,939.261133,939.261133
88,1.0,9540.124728,9540.124728


In [11]:
sx = data['CO2'].sum()/data['F'].sum()  # 总CO2排放/总燃料消耗
sy = 0  # 产出方向调整系数设为0
# 定义并求解DEA模型
def MB_CRSL3():

    # model = DEA.DEA(data,sent = "K L E=Y:CO2",  orient="L", rts=RTS_VRS, baseindex=None,refindex=None)
    model =MB.MB(data,sent = "K L+F=E:CO2", sx=[[0,0,sx]], sy=[[sy]], \
                    rts=RTS_CRS,baseindex=None,refindex=None,level=3)
    res, info = model.optimize("mosek", "1")  # 使用Mosek求解器
    return res
MB_CRSL3 = MB_CRSL3()
MB_CRSL3

,optimization_status,obj,best of Undesirable
0,1.0,1560.103792,1560.103792
1,1.0,3617.484002,3617.484002
2,1.0,17092.312335,17092.312335
3,1.0,13905.609612,13905.609612
4,1.0,20906.091263,20906.091263
...,...,...,...
85,1.0,9322.480651,9322.480651
86,1.0,4294.522427,4294.522427
87,1.0,906.062583,906.062583
88,1.0,7102.157964,7102.157964


In [12]:
sx = data['CO2'].sum()/data['F'].sum()  # 总CO2排放/总燃料消耗
sy = 0  # 产出方向调整系数设为0
# 定义并求解DEA模型
def MB_CRSL4():

    # model = DEA.DEA(data,sent = "K L E=Y:CO2",  orient="L", rts=RTS_VRS, baseindex=None,refindex=None)
    model =MB.MB(data,sent = "K L+F=E:CO2", sx=[[0,0,sx]], sy=[[sy]], \
                    rts=RTS_CRS,baseindex=None,refindex=None,level=4)
    res, info = model.optimize("mosek", "1")  # 使用Mosek求解器
    return res
MB_CRSL4 = MB_CRSL4()
MB_CRSL4

,optimization_status,obj,best of Undesirable
0,1.0,-0.000000,-0.000000
1,1.0,-0.000000,-0.000000
2,1.0,3247.575108,3247.575108
3,1.0,-0.000000,-0.000000
4,1.0,206.252201,206.252201
...,...,...,...
85,1.0,-0.000000,-0.000000
86,1.0,-0.000000,-0.000000
87,1.0,-0.000000,-0.000000
88,1.0,38.178262,38.178262


In [13]:
MB_VRSL1

,optimization_status,obj,best of Undesirable
0,1.0,1592.340000,1592.340000
1,1.0,4794.003224,4794.003224
2,1.0,21825.209573,21825.209573
3,1.0,17951.812421,17951.812421
4,1.0,27910.765567,27910.765567
...,...,...,...
85,1.0,12202.430819,12202.430819
86,1.0,4923.135529,4923.135529
87,1.0,1336.275945,1336.275945
88,1.0,11625.400000,11625.400000


In [14]:
MB_VRSL1 = MB_VRSL1.drop(columns = ['optimization_status','obj'])
MB_CRSL2 = MB_CRSL2.drop(columns = ['optimization_status','obj'])
MB_CRSL3 = MB_CRSL3.drop(columns = ['optimization_status','obj'])
MB_CRSL4 = MB_CRSL4.drop(columns = ['optimization_status','obj'])

MB_VRSL1 = MB_VRSL1.rename(columns = {'best of Undesirable':'B1'})
MB_CRSL2 = MB_CRSL2.rename(columns = {'best of Undesirable':'B2'})
MB_CRSL3 = MB_CRSL3.rename(columns = {'best of Undesirable':'B3'})
MB_CRSL4 = MB_CRSL4.rename(columns = {'best of Undesirable':'B4'})

MB_VRSL1['B1'] = abs(MB_VRSL1['B1'])
MB_CRSL2['B2'] = abs(MB_CRSL2['B2'])
MB_CRSL3['B3'] = abs(MB_CRSL3['B3'])
MB_CRSL4['B4'] = abs(MB_CRSL4['B4'])



MB_CRS = pd.concat([yearid, MB_VRSL1], axis=1)  # 合并基础信息
MB_CRS = pd.concat([MB_CRS,MB_CRSL2],axis=1)
MB_CRS = pd.concat([MB_CRS,MB_CRSL3],axis=1)
MB_CRS = pd.concat([MB_CRS,MB_CRSL4],axis=1)

In [15]:
MB_CRS['OE'] = abs(MB_CRS['B4'] )/ abs(MB_CRS['CO2'] )
MB_CRS['TE'] = abs(MB_CRS['B1']) / abs(MB_CRS['CO2'] )
MB_CRS['PIAE'] = abs(MB_CRS['B2']) / abs(MB_CRS['B1'] )
MB_CRS['NPIAE'] = MB_CRS['B3'] / MB_CRS['B2'] # 非投入配置效率 = B3/B2
MB_CRS['NPOAE'] = MB_CRS['B4'] / MB_CRS['B3'] # 非产出配置效率 = B4/B3（理论公式用B5/B4，因无Y_NP故B5=B4）


cat_size_order

CategoricalDtype(categories=['OE', 'TE', 'PIAE', 'NPIAE', 'NPOAE'], ordered=True, categories_dtype=object)

In [16]:
MB_CRS.query('year==2017')#.sort_values('MQ')

,year,省份,CO2,B1,B2,B3,B4,OE,TE,PIAE,NPIAE,NPOAE
60,2017,北京,1328.24,1328.240000,1328.240000,1328.240000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000
61,2017,天津,4927.63,4598.260410,4481.460618,3463.065632,0.000000,0.000000,0.933159,0.974599,0.772754,0.000000
62,2017,河北,26802.07,26802.070000,26802.070000,19424.925127,4450.323776,0.166044,1.000000,1.000000,0.724755,0.229104
63,2017,山西,25022.67,19386.721039,19300.733112,14923.800994,0.000000,0.000000,0.774766,0.995565,0.773225,0.000000
64,2017,内蒙古,44495.77,30816.956671,30401.168911,23005.821700,319.434945,0.007179,0.692582,0.986508,0.756741,0.013885
65,2017,辽宁,16003.93,11327.545818,10956.516576,9107.111712,584.374469,0.036514,0.707798,0.967245,0.831205,0.064167
66,2017,吉林,7850.81,4140.344913,3681.375507,3663.071899,0.000000,0.000000,0.527378,0.889147,0.995028,0.000000
67,2017,黑龙江,8633.52,5529.425284,4684.080062,4683.769285,0.000000,0.000000,0.640460,0.847119,0.999934,0.000000
68,2017,上海,7138.07,7138.070000,6735.574188,5113.729785,0.000000,0.000000,1.000000,0.943613,0.759212,0.000000
69,2017,江苏,37054.37,37054.370000,37054.370000,26464.926920,0.000000,0.000000,1.000000,1.000000,0.714219,0.000000


In [17]:
MB_CRS3 = MB_CRS.pivot_table(
    values=['OE', 'TE', 'PIAE', 'NPIAE', 'NPOAE'],  # 效率指标
    index=dmuname,  # 行索引为省份
    columns='year',  # 列索引为年份
    aggfunc='mean',  # 计算均值
    fill_value=None,
    margins=False,
    dropna=True,
    margins_name='All',
    observed=False,
)    [['OE',	'TE',	'PIAE',	'NPIAE',	'NPOAE']]

In [18]:
MB_CRS3

OE                            TE                          PIAE  \
year      2015      2016      2017      2015      2016      2017      2015   
省份                                                                           
上海    0.031004  0.018273  0.000000  1.000000  0.977622  1.000000  0.964068   
云南    0.032409  0.000000  0.012322  0.582192  0.574859  0.557999  0.906367   
内蒙古   0.005232  0.006365  0.007179  0.707959  0.695773  0.692582  0.985751   
北京    0.000000  0.000000  0.000000  1.000000  1.000000  1.000000  1.000000   
吉林    0.000000  0.000000  0.000000  0.521593  0.528705  0.527378  0.903726   
四川    0.083117  0.120015  0.144551  0.644366  0.697759  0.710887  0.979021   
天津    0.000000  0.000000  0.000000  0.937162  1.000000  0.933159  0.991293   
宁夏    0.000000  0.000239  0.003284  1.000000  0.963093  1.000000  0.870653   
安徽    0.000000  0.000000  0.000000  0.895015  0.908827  0.925465  0.997950   
山东    0.014746  0.007639  0.008902  0.955517  1.000000  1.000000  0.984955   
山西    0.000000  0.000000  0.000000  0.814743  0.813181  0.774766  0.997363   
广东    0.000000  0.000000  0.000000  0.960185  0.979635  1.000000  0.856065   
广西    0.000000  0.000000  0.003472  0.702276  0.685174  0.687119  0.921632   
新疆    0.000000  0.000000  0.000000  0.857847  0.837050  0.822347  0.999800   
江苏    0.000000  0.000000  0.000000  0.994871  1.000000  1.000000  0.984960   
江西    0.011499  0.000000  0.000000  0.807207  0.847555  0.862762  0.874398   
河北    0.135243  0.151460  0.166044  0.908897  0.899945  1.000000  0.999885   
河南    0.000000  0.000000  0.000000  0.877889  0.865403  0.874602  0.907893   
浙江    0.000000  0.000000  0.000000  1.000000  1.000000  1.000000  1.000000   
海南    0.000000  0.000000  0.000000  1.000000  1.000000  1.000000  0.980608   
湖北    0.032364  0.028567  0.007716  0.844753  0.865934  0.876096  0.869077   
湖南    0.018901  0.097623  0.098146  0.786299  0.804697  0.839009  0.876673   
甘肃    0.000000  0.000000  0.000000  0.772674  0.779346  0.764100  0.876281   
福建    0.000000  0.000000  0.000000  0.836522  0.793791  0.828802  0.996084   
贵州    0.000000  0.000000  0.000000  0.905482  0.905058  0.850053  0.904821   
辽宁    0.044763  0.052623  0.036514  0.701138  0.733603  0.707798  0.953157   
重庆    0.000000  0.000000  0.000000  0.765456  0.739204  0.759440  0.982187   
陕西    0.000000  0.000000  0.000000  0.948327  0.984612  0.973494  0.967692   
青海    0.000000  0.000000  0.000000  1.000000  0.984890  0.935433  0.594048   
黑龙江   0.000000  0.000000  0.000000  0.664436  0.645498  0.640460  0.853666   

                             NPIAE                         NPOAE            \
year      2016      2017      2015      2016      2017      2015      2016   
省份                                                                           
上海    0.961904  0.943613  0.777020  0.777552  0.759212  0.041388  0.024991   
云南    0.845118  0.852840  1.000000  1.000000  1.000000  0.061417  0.000000   
内蒙古   0.983007  0.986508  0.759861  0.768336  0.756741  0.009866  0.012111   
北京    1.000000  1.000000  0.979755  0.999656  1.000000  0.000000  0.000000   
吉林    0.899259  0.889147  1.000000  1.000000  0.995028  0.000000  0.000000   
四川    0.980598  0.984439  1.000000  1.000000  1.000000  0.131755  0.175404   
天津    0.946106  0.974599  0.761213  0.766140  0.772754  0.000000  0.000000   
宁夏    0.971211  0.820628  0.730112  0.753031  0.744451  0.000000  0.000339   
安徽    0.999045  0.999765  0.777038  0.770951  0.755448  0.000000  0.000000   
山东    1.000000  0.961953  0.746130  0.727224  0.770852  0.020999  0.010505   
山西    0.992665  0.995565  0.776656  0.780629  0.773225  0.000000  0.000000   
广东    0.842010  0.854480  0.846570  0.847264  0.811921  0.000000  0.000000   
广西    0.909216  0.898854  1.000000  1.000000  0.999914  0.000000  0.000000   
新疆    0.998285  0.998491  0.750504  0.758858  0.760210  0.000000  0.000000   
江苏    1.000000  1.000000  0.742164  0.728928  0.714219  0.000000  0.000000   
江西    0.986949  0.999865  0.97890

In [19]:
MB_CRS3[['OE']].to_markdown("MB_CRS_1.md")  # 整体效率结果

MB_CRS3[['TE','PIAE']].to_markdown("MB_CRS_23.md")  # 技术和投入配置效率

MB_CRS3[['NPIAE','NPOAE']].to_markdown("MB_CRS_45.md")  # 非投入和非产出配置效率

In [21]:
MB_CRS3[['OE']]

OE                    
year      2015      2016      2017
省份                                
上海    0.031004  0.018273  0.000000
云南    0.032409  0.000000  0.012322
内蒙古   0.005232  0.006365  0.007179
北京    0.000000  0.000000  0.000000
吉林    0.000000  0.000000  0.000000
四川    0.083117  0.120015  0.144551
天津    0.000000  0.000000  0.000000
宁夏    0.000000  0.000239  0.003284
安徽    0.000000  0.000000  0.000000
山东    0.014746  0.007639  0.008902
山西    0.000000  0.000000  0.000000
广东    0.000000  0.000000  0.000000
广西    0.000000  0.000000  0.003472
新疆    0.000000  0.000000  0.000000
江苏    0.000000  0.000000  0.000000
江西    0.011499  0.000000  0.000000
河北    0.135243  0.151460  0.166044
河南    0.000000  0.000000  0.000000
浙江    0.000000  0.000000  0.000000
海南    0.000000  0.000000  0.000000
湖北    0.032364  0.028567  0.007716
湖南    0.018901  0.097623  0.098146
甘肃    0.000000  0.000000  0.000000
福建    0.000000  0.000000  0.000000
贵州    0.000000  0.000000  0.000000
辽宁    0.044763  0.052623  0.036514
重庆    0.000000  0.000000  0.000000
陕西    0.000000  0.000000  0.000000
青海    0.000000  0.000000  0.000000
黑龙江   0.000000  0.000000  0.000000

In [22]:
MB_CRS3[['TE','PIAE']]

TE                          PIAE                    
year      2015      2016      2017      2015      2016      2017
省份                                                              
上海    1.000000  0.977622  1.000000  0.964068  0.961904  0.943613
云南    0.582192  0.574859  0.557999  0.906367  0.845118  0.852840
内蒙古   0.707959  0.695773  0.692582  0.985751  0.983007  0.986508
北京    1.000000  1.000000  1.000000  1.000000  1.000000  1.000000
吉林    0.521593  0.528705  0.527378  0.903726  0.899259  0.889147
四川    0.644366  0.697759  0.710887  0.979021  0.980598  0.984439
天津    0.937162  1.000000  0.933159  0.991293  0.946106  0.974599
宁夏    1.000000  0.963093  1.000000  0.870653  0.971211  0.820628
安徽    0.895015  0.908827  0.925465  0.997950  0.999045  0.999765
山东    0.955517  1.000000  1.000000  0.984955  1.000000  0.961953
山西    0.814743  0.813181  0.774766  0.997363  0.992665  0.995565
广东    0.960185  0.979635  1.000000  0.856065  0.842010  0.854480
广西    0.702276  0.685174  0.687119  0.921632  0.909216  0.898854
新疆    0.857847  0.837050  0.822347  0.999800  0.998285  0.998491
江苏    0.994871  1.000000  1.000000  0.984960  1.000000  1.000000
江西    0.807207  0.847555  0.862762  0.874398  0.986949  0.999865
河北    0.908897  0.899945  1.000000  0.999885  0.999961  1.000000
河南    0.877889  0.865403  0.874602  0.907893  0.897741  0.904705
浙江    1.000000  1.000000  1.000000  1.000000  1.000000  1.000000
海南    1.000000  1.000000  1.000000  0.980608  0.800122  0.746078
湖北    0.844753  0.865934  0.876096  0.869077  0.872286  0.887049
湖南    0.786299  0.804697  0.839009  0.876673  0.886557  0.875293
甘肃    0.772674  0.779346  0.764100  0.876281  0.876260  0.872488
福建    0.836522  0.793791  0.828802  0.996084  0.991505  0.999167
贵州    0.905482  0.905058  0.850053  0.904821  0.919709  0.983028
辽宁    0.701138  0.733603  0.707798  0.953157  0.958304  0.967245
重庆    0.765456  0.739204  0.759440  0.982187  0.978128  0.978274
陕西    0.948327  0.984612  0.973494  0.967692  0.975645  0.987207
青海    1.000000  0.984890  0.935433  0.594048  0.700787  0.702895
黑龙江   0.664436  0.645498  0.640460  0.853666  0.849608  0.847119

In [23]:
MB_CRS3[['NPIAE','NPOAE']]

NPIAE                         NPOAE                    
year      2015      2016      2017      2015      2016      2017
省份                                                              
上海    0.777020  0.777552  0.759212  0.041388  0.024991  0.000000
云南    1.000000  1.000000  1.000000  0.061417  0.000000  0.025893
内蒙古   0.759861  0.768336  0.756741  0.009866  0.012111  0.013885
北京    0.979755  0.999656  1.000000  0.000000  0.000000  0.000000
吉林    1.000000  1.000000  0.995028  0.000000  0.000000  0.000000
四川    1.000000  1.000000  1.000000  0.131755  0.175404  0.206553
天津    0.761213  0.766140  0.772754  0.000000  0.000000  0.000000
宁夏    0.730112  0.753031  0.744451  0.000000  0.000339  0.005376
安徽    0.777038  0.770951  0.755448  0.000000  0.000000  0.000000
山东    0.746130  0.727224  0.770852  0.020999  0.010505  0.012005
山西    0.776656  0.780629  0.773225  0.000000  0.000000  0.000000
广东    0.846570  0.847264  0.811921  0.000000  0.000000  0.000000
广西    1.000000  1.000000  0.999914  0.000000  0.000000  0.005622
新疆    0.750504  0.758858  0.760210  0.000000  0.000000  0.000000
江苏    0.742164  0.728928  0.714219  0.000000  0.000000  0.000000
江西    0.978904  0.847401  0.802043  0.016644  0.000000  0.000000
河北    0.783236  0.788324  0.724755  0.190002  0.213498  0.229104
河南    0.847744  0.858678  0.850379  0.000000  0.000000  0.000000
浙江    0.760612  0.761281  0.757532  0.000000  0.000000  0.000000
海南    0.769098  0.840424  0.821716  0.000000  0.000000  0.000000
湖北    0.963361  0.956263  0.932844  0.045760  0.039549  0.010643
湖南    1.000000  1.000000  0.999872  0.027419  0.136840  0.133662
甘肃    0.994033  0.998465  0.999801  0.000000  0.000000  0.000000
福建    0.815669  0.839739  0.806632  0.000000  0.000000  0.000000
贵州    0.901209  0.877935  0.818457  0.000000  0.000000  0.000000
辽宁    0.852885  0.841416  0.831205  0.078534  0.088961  0.064167
重庆    0.896678  0.877672  0.866361  0.000000  0.000000  0.000000
陕西    0.797753  0.785499  0.773886  0.000000  0.000000  0.000000
青海    0.987354  0.953515  0.964655  0.000000  0.000000  0.000000
黑龙江   0.999874  0.999897  0.999934  0.000000  0.000000  0.000000